In [ ]:
# parameters
p_dt=""


# Upload Data from Bronze to Silver



## 1. Service functions

In [ ]:
from pyspark.sql.functions import lit, col, expr
from delta.tables import DeltaTable
from datetime import datetime
import json
import uuid



def set_opdate_status(p_dt, p_status):
    """Set the status of the operating day"""
    # if we set the status to "O" then all others should be closed to "C"
    if p_status=='O':
        spark.sql("""
                    UPDATE psh_dwh_lh.dbo.calendar
                    SET STATUS='C'
                    WHERE  /*OPDATE = :opdate and*/ STATUS=:status
                """, {"status": p_status})

    # Set the status for the current operation day
    spark.sql("""
                UPDATE psh_dwh_lh.dbo.calendar
                SET STATUS=:status
                WHERE  OPDATE = :opdate
            """, {"opdate": p_dt, "status": p_status})

def init_upload(p_opdate):
    """
        Create a record for the start of the data upload process and return its upload_id
    """
    # Generate a unique upload ID using Python
    u_id = str(uuid.uuid4())
    # Determine the current time for the upload timestamp
    current_ts = datetime.now()
    spark.sql(
        """
            INSERT INTO  psh_dwh_lh.dbo.ev2_silver_upload_grn
            (upload_id, opdate, upload_status, upload_at)
            VALUES
            (:upload_id, :opdate, :upload_status, :upload_at )
        """, 
        { 
            "upload_id": u_id, 
            "opdate": p_dt, 
            "upload_status": 'STARTED', 
            "upload_at": current_ts 
        }
    )
    print(f"Upload initiated {u_id} for date {p_opdate}")
    return u_id

def close_upload(p_upload_id):
    """
        Close (finish) the record for the data upload process by upload_id
    """

    # Determine the current time for the upload timestamp
    current_ts = datetime.now()
    spark.sql(
        """
            UPDATE  psh_dwh_lh.dbo.ev2_silver_upload_grn
                SET upload_status = :upload_status, 
                    upload_at = :upload_at
            WHERE  upload_id = :upload_id       
          
        """, 
        { 
            "upload_id": p_upload_id, 
            "upload_status": 'SUCCESS', 
            "upload_at": current_ts 
        }
    )
    print(f"Finished successfully the data upload process {p_upload_id} ")
   


## 2. Main handler

### 2.1. Create a new record about the start of processing

In [ ]:
up_opdate = p_dt
continue_processing = True
result_data = {
    "status": "Error",
    "error": None,
    "upload_id": None,
    "upload_opdate": None    
}

if up_opdate == "" or not up_opdate:
    continue_processing = False
    result_data["status"] = "Error"
    result_data["error"] = "No open days found OPDATE" 

if not continue_processing :
    mssparkutils.notebook.exit(json.dumps(result_data))

try:
    up_id=init_upload(up_opdate)
    result_data["upload_id"] = up_id
    result_data["upload_opdate"] = up_opdate

except Exception as e:
    print(f"Error processing : {e}")
    continue_processing = False
    result_data["status"] = "Error"
    result_data["error"] = str(e)     

### 2.2. Upload the terminal dimension

## 

In [ ]:
if not continue_processing :
    mssparkutils.notebook.exit(json.dumps(result_data))

try:

    # Upload the terminal dimension
    print("Upload the terminal dimension")
    trm_up_id=result_data["upload_id"] 
    trm_idt_ts = datetime.now()
    target_table="psh_dwh_lh.dbo.ev2_silver_terminals_dim"
    source_table="psh_dwh_lh.dbo.terminals_d"
    spark.sql(f"""
            MERGE INTO {target_table} AS target
            USING {source_table} AS source
            ON target.terminal_id = source.TERMINAL_ID
            WHEN MATCHED THEN
                UPDATE SET 
                    target.terminal_name = source.TERMINAL_NAME, 
                    target.isactive = source.ISACTIVE,
                    target.upload_id=:trm_up_id,
                    target.idt = :idt_ts
                    
            WHEN NOT MATCHED THEN
                INSERT (terminal_id, terminal_name, isactive, upload_id, idt )
                VALUES ( source.TERMINAL_ID, source.TERMINAL_NAME, source.ISACTIVE, 
                         :trm_up_id, :idt_ts)
        """
        , {
            "trm_up_id": trm_up_id, 
            "idt_ts":trm_idt_ts 
        }
    )
    print("Upload the terminal dimension-OK")
except Exception as e:
    print(f"Error uploading terminal dimension : {e}")
    continue_processing = False
    result_data["status"] = "Error"
    result_data["error"] = str(e)
   

### 2.3. Upload customer dimension

In [ ]:
if not continue_processing :
    mssparkutils.notebook.exit(json.dumps(result_data))

try:


    print("Upload customer dimension")
    
    # отримую параметри завантаження
    cu_up_id = result_data["upload_id"] 
    cu_up_opdate = result_data["upload_opdate"] 
    cu_idt_ts = datetime.now()

    cu_target_table="psh_dwh_lh.dbo.ev2_silver_customers_dim"
    cu_source_table="psh_dwh_lh.dbo.ev2_file_body"

    print("Upload customer dimension - forming customer buffer from transactions....")
    # make a dataframe of customers from the file body table by transactions using cu_up_opdate

    cu_df_buff = spark.sql(f"""
            SELECT tr.NAME as cust_name, tr.PASSPORT as cust_doc
            FROM {cu_source_table} tr
            WHERE tr.OPDATE = :p_opdate
       """,
       {
        "p_opdate": cu_up_opdate
       }
    )
    print("Upload customer dimension - forming customer buffer from transactions - OK")
    
    # do a merge into the target table psh_dwh_lh.dbo.ev2_silver_customers_dim
    print("Upload customer dimension - making merge.....")
    # 1. First, we remove duplicates from the input stream (if one customer made 2 transactions)
    # We leave unique Name + Passport pairs
    cu_df_distinct = cu_df_buff.select("cust_name", "cust_doc").distinct()

    # 2. Generate UUID in advance for the entire set
    # Now each ROW in cu_df_final has its own fixed ID
    cu_df_final = cu_df_distinct.withColumn("new_cust_id", expr("uuid()"))

    # 3. Get a reference to the target Delta table
    delta_target = DeltaTable.forName(spark, cu_target_table)

    # 4. Execute the MERGE
    print("Upload customer dimension - updating.....")
    delta_target.alias("target").merge(
        cu_df_final.alias("source"),
        "target.cust_name = source.cust_name AND target.cust_doc = source.cust_doc"
    ).whenMatchedUpdate(set = {
        "upload_id": lit(cu_up_id),
        "idt": lit(cu_idt_ts)
    }).whenNotMatchedInsert(values = {
        "cust_id": "source.new_cust_id", # We use an already generated ID
        "cust_name": "source.cust_name",
        "cust_doc": "source.cust_doc",
        "cust_status": lit("ACTIVE"),
        "upload_id": lit(cu_up_id),
        "idt": lit(cu_idt_ts)
    }).execute()

    print("Upload customer dimension - making merge - OK")
    print("Upload customer dimension-OK")
except Exception as e:
    print(f"Upload customer dimension-ERROR : {e}")
    continue_processing = False
    result_data["status"] = "Error"
    result_data["error"] = str(e)

## 2.4. Upload transactions

In [ ]:
if not continue_processing :
    mssparkutils.notebook.exit(json.dumps(result_data))

try:
    print("Upload transactions....")
    trn_up_id=result_data["upload_id"] 
    trn_up_opdate = result_data["upload_opdate"] 
    trn_idt_ts = datetime.now()
    target_table="psh_dwh_lh.dbo.ev2_silver_transactions"
    source_table="psh_dwh_lh.dbo.ev2_file_body"
    spark.sql(f"""
            MERGE INTO {target_table} AS target
            USING {source_table} AS source
            ON target.tx_id = source.TX_ID 
                AND target.file_id = source.FILE_ID 
                AND target.opdate = source.OPDATE
                AND source.OPDATE = :p_opdate
            WHEN MATCHED THEN
                UPDATE SET 
                    target.terminal_id = source.TERMINAL_ID, 
                    target.opdate = source.OPDATE,
                    target.amount = source.AMOUNT,
                    target.charge = source.CHARGE,
                    target.status = 'MODIFIED',
                    target.file_name = source.FILE_NAME,
                    target.file_id = source.FILE_ID,
                    target.upload_id=:trn_up_id,
                    target.idt = :idt_ts
            WHEN NOT MATCHED THEN
                INSERT (tx_id, terminal_id, opdate, amount, charge, status,file_name, file_id, upload_id, idt )
                VALUES (source.TX_ID, source.TERMINAL_ID, source.OPDATE, source.AMOUNT, source.CHARGE, 'ACCEPTED',
                         source.FILE_NAME, source.FILE_ID, :trn_up_id, :idt_ts)
        """
        , {
            "trn_up_id": trn_up_id, 
            "idt_ts": trn_idt_ts,
            "p_opdate": trn_up_opdate
        }
    )
    print("Upload transactions - OK")
except Exception as e:
    print(f"Upload transactions ERROR : {e}")
    continue_processing = False
    result_data["status"] = "Error"
    result_data["error"] = str(e)
   

### 2.5. Finish uploading into Silver level

### 2.5.1. Automatically switch to the new date

In [ ]:
def set_opdate_status(p_dt, p_status):
    """Set operation day status"""
    # if we set status "O" then all others must be closed in "C"
    if p_status=='O':
        spark.sql("""
                    UPDATE psh_dwh_lh.dbo.calendar
                    SET STATUS='C'
                    WHERE  /*OPDATE = :opdate and*/ STATUS=:status
                """, {"status": p_status})

    # Set the status of the current operation day
    spark.sql("""
                UPDATE psh_dwh_lh.dbo.calendar
                SET STATUS=:status
                WHERE  OPDATE = :opdate
            """, {"opdate": p_dt, "status": p_status})

# 1. Get the first available day where there are transactions without cust_id
df_next_day = spark.sql("""
    SELECT A.opdate
    FROM psh_dwh_lh.dbo.ev2_silver_transactions A
    WHERE A.cust_id IS NULL
    GROUP BY A.opdate
    ORDER BY A.opdate
    LIMIT 1
""")

# 2. Check if we found anything
if df_next_day.count() > 0:
    # Get the date value from the first row
    next_opdate = df_next_day.collect()[0]['opdate']
    print(f"Found next day for processing: {next_opdate}")
    # Run the function in the background
    pa_status = "O" 
    set_opdate_status(next_opdate, pa_status)
    
else:
    print("All days processed. No new data available.")
    mssparkutils.notebook.exit(json.dumps({"status": "SUCCESS", "msg": "No data to process"}))

### 2.5.2 Finish current uploading 

In [ ]:

if not continue_processing :
    mssparkutils.notebook.exit(json.dumps(result_data))

try:
    
    fin_up_id = result_data["upload_id"] = up_id
    fin_up_opdate = result_data["upload_opdate"]
    print(f"Finish uploading into Silver level  {fin_up_id} за {fin_up_opdate} .....")
    close_upload(  fin_up_id )
    result_data["status"] = "SUCCESS"
    print(f"Finish uploading into Silver level  {fin_up_id} за {fin_up_opdate} - OK")
except Exception as e:
    print(f"Помилка обробки : {e}")
    continue_processing = False
    result_data["status"] = "Error"
    result_data["error"] = str(e)    

mssparkutils.notebook.exit(json.dumps(result_data))
